In [1]:
# Cell 2: Install the only required notebook dependency

%pip install requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cell 3: Imports and configuration

from __future__ import annotations

import copy
import json
import textwrap
from collections import Counter
from pathlib import Path
from typing import Any, Dict, List, Optional

import requests
from pprint import pprint

# ---------------------------
# User-editable configuration
# ---------------------------

OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_GENERATE_URL = f"{OLLAMA_BASE_URL}/api/generate"
OLLAMA_TAGS_URL = f"{OLLAMA_BASE_URL}/api/tags"

# Change this if your local Ollama model tag is different.
OLLAMA_MODEL = "mistral:7b"

# Type your input path here after pasting into your notebook.
CLASSIFIED_TRANSCRIPT_PATH = Path("transcription_outputs/classified_segments.json")

# Optional:
# Leave as None to use the embedded default SOAP template below.
# Or point it to a JSON file on disk if you want to swap in another chart structure.
OPTIONAL_TEMPLATE_PATH = None
# Example:
# OPTIONAL_TEMPLATE_PATH = Path("/absolute/path/to/your/custom_chart_template.json")

OUTPUT_DIR = Path("./soap_note_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional safety knob for very long transcripts.
# Leave as None to send the full transcript.
MAX_TRANSCRIPT_CHARS = None

In [3]:
# Cell 4: Embedded default SOAP template

DEFAULT_SOAP_TEMPLATE = {
    "schema_version": "v1",
    "note_type": "soap_note",
    "encounter": {
        "encounter_id": None,
        "date": None,
        "clinician": None,
        "patient": None
    },
    "subjective": {
        "chief_complaint": None,
        "hpi_summary": None,
        "medications_mentioned": [],
        "allergies_mentioned": [],
        "relevant_history": [],
        "patient_reported_symptoms": [],
        "patient_denied_symptoms": []
    },
    "objective": {
        "vitals": {
            "bp": None,
            "hr": None,
            "temp": None,
            "rr": None,
            "spo2": None,
            "weight": None
        },
        "exam_findings": [],
        "tests_or_results_discussed": []
    },
    "assessment": {
        "clinician_impression": [],
        "problems_addressed": []
    },
    "plan": {
        "medications_started_or_changed": [],
        "tests_ordered": [],
        "referrals": [],
        "patient_instructions": [],
        "follow_up": None,
        "return_precautions": []
    },
    "evidence": {
        "subjective_segment_ids": [],
        "objective_segment_ids": [],
        "assessment_segment_ids": [],
        "plan_segment_ids": []
    },
    "quality": {
        "missing_sections": [],
        "uncertain_fields": [],
        "notes": None
    }
}

print("Default template loaded.")

Default template loaded.


In [4]:
# Cell 5: Helper functions for loading, normalization, validation, and rendering

def load_json(path: Path) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(path: Path, payload: Dict[str, Any]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)


def safe_float(value: Any, default: float = 0.0) -> float:
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def normalize_speaker(raw_value: Any) -> str:
    value = str(raw_value or "").strip().lower()

    doctor_aliases = {
        "doctor", "dr", "dr.", "physician", "provider", "clinician", "speaker_doctor"
    }
    patient_aliases = {
        "patient", "pt", "speaker_patient"
    }

    if value in doctor_aliases:
        return "doctor"
    if value in patient_aliases:
        return "patient"
    return "unknown"


def normalize_segments(payload: Dict[str, Any]) -> List[Dict[str, Any]]:
    raw_segments = (
        payload.get("segments")
        or payload.get("classified_segments")
        or payload.get("utterances")
        or []
    )

    normalized = []

    for idx, seg in enumerate(raw_segments):
        seg_id = seg.get("id", idx)

        speaker = normalize_speaker(
            seg.get("speaker")
            or seg.get("role")
            or seg.get("predicted_role") #added this into it 202603121743
            or seg.get("label")
            or seg.get("speaker_label")
            or seg.get("classified_speaker")
        )

        text = " ".join(str(seg.get("text", "")).split()).strip()
        if not text:
            continue

        normalized.append(
            {
                "id": str(seg_id),
                "start": safe_float(seg.get("start")),
                "end": safe_float(seg.get("end")),
                "speaker": speaker,
                "text": text,
            }
        )

    return normalized


def ensure_required_meta_sections(template: Dict[str, Any]) -> Dict[str, Any]:
    """
    This lets you swap in a different chart template later while still keeping
    encounter/evidence/quality fields available for this notebook's workflow.
    """
    template = copy.deepcopy(template)

    if "encounter" not in template:
        template["encounter"] = copy.deepcopy(DEFAULT_SOAP_TEMPLATE["encounter"])

    if "evidence" not in template:
        template["evidence"] = copy.deepcopy(DEFAULT_SOAP_TEMPLATE["evidence"])

    if "quality" not in template:
        template["quality"] = copy.deepcopy(DEFAULT_SOAP_TEMPLATE["quality"])

    return template


def load_note_template(optional_template_path: Optional[Path]) -> Dict[str, Any]:
    if optional_template_path is None:
        return ensure_required_meta_sections(copy.deepcopy(DEFAULT_SOAP_TEMPLATE))

    template = load_json(optional_template_path)
    return ensure_required_meta_sections(template)


def fill_encounter_defaults(
    template: Dict[str, Any],
    transcript_payload: Dict[str, Any],
    source_path: Path
) -> Dict[str, Any]:
    template = copy.deepcopy(template)

    encounter = template.get("encounter", {})
    encounter["encounter_id"] = (
        encounter.get("encounter_id")
        or transcript_payload.get("encounter_id")
        or transcript_payload.get("visit_id")
        or source_path.stem
    )
    encounter["date"] = (
        encounter.get("date")
        or transcript_payload.get("date")
        or transcript_payload.get("encounter_date")
    )
    encounter["clinician"] = (
        encounter.get("clinician")
        or transcript_payload.get("clinician")
        or transcript_payload.get("doctor")
        or transcript_payload.get("provider")
    )
    encounter["patient"] = (
        encounter.get("patient")
        or transcript_payload.get("patient")
        or transcript_payload.get("patient_name")
    )

    template["encounter"] = encounter
    return template


def build_transcript_for_llm(
    segments: List[Dict[str, Any]],
    max_chars: Optional[int] = None
) -> str:
    lines = []

    for seg in segments:
        line = (
            f"[segment_id={seg['id']} | speaker={seg['speaker']} | "
            f"start={seg['start']:.2f} | end={seg['end']:.2f}] {seg['text']}"
        )
        lines.append(line)

    transcript_text = "\n".join(lines)

    if max_chars is not None:
        transcript_text = transcript_text[:max_chars]

    return transcript_text


def template_to_json_schema(template: Any) -> Dict[str, Any]:
    """
    A small permissive JSON-schema builder.
    It keeps object keys fixed, while allowing simple scalar values or lists.
    """

    if isinstance(template, dict):
        return {
            "type": "object",
            "properties": {
                key: template_to_json_schema(value)
                for key, value in template.items()
            },
            "required": list(template.keys()),
            "additionalProperties": False,
        }

    if isinstance(template, list):
        return {
            "type": "array",
            "items": {
                "type": ["string", "number", "integer", "boolean", "null", "object"]
            },
        }

    return {
        "type": ["string", "number", "integer", "boolean", "null"]
    }


def build_charting_prompt(
    note_template: Dict[str, Any],
    transcript_text: str
) -> str:
    template_str = json.dumps(note_template, indent=2, ensure_ascii=False)

    return f"""
You are converting a classified doctor-patient conversation into a structured chart note.

Important rules:
1. Use only information supported by the transcript.
2. Do not invent diagnoses, vitals, medications, allergies, tests, follow-up details, or instructions.
3. If a field is not supported, leave it null or use an empty list.
4. Keep wording concise and chart-like.
5. Fill the evidence section using only segment ids that appear in the transcript.
6. Evidence should be section-level:
   - evidence.subjective_segment_ids
   - evidence.objective_segment_ids
   - evidence.assessment_segment_ids
   - evidence.plan_segment_ids
7. quality.uncertain_fields should contain dotted field names such as:
   - subjective.hpi_summary
   - objective.vitals.bp
   - plan.follow_up
8. quality.missing_sections should contain any major sections that are mostly unsupported:
   - subjective
   - objective
   - assessment
   - plan
9. Output exactly one JSON object and nothing else.
10. Do not wrap your answer in markdown.

Use this exact target structure:
{template_str}

Here is the classified transcript:
{transcript_text}
""".strip()


def call_ollama_generate_json(
    model: str,
    prompt: str,
    json_schema: Dict[str, Any],
    timeout_seconds: int = 600,
) -> Dict[str, Any]:
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "format": json_schema,
        "options": {
            "temperature": 0
        }
    }

    response = requests.post(
        OLLAMA_GENERATE_URL,
        json=payload,
        timeout=timeout_seconds,
    )
    response.raise_for_status()

    raw = response.json()
    response_text = raw.get("response", "").strip()

    if not response_text:
        raise ValueError("Ollama returned an empty response.")

    try:
        parsed = json.loads(response_text)
    except json.JSONDecodeError as e:
        raise ValueError(f"Model response was not valid JSON.\n\nResponse text:\n{response_text}") from e

    return {
        "raw_ollama_response": raw,
        "parsed_json": parsed,
        "prompt": prompt,
    }


def coerce_to_template(template: Any, candidate: Any) -> Any:
    """
    Merge model output onto the template.
    This keeps the final shape stable even if the model misses fields.
    """

    if isinstance(template, dict):
        out = {}
        candidate = candidate if isinstance(candidate, dict) else {}
        for key, value in template.items():
            out[key] = coerce_to_template(value, candidate.get(key))
        return out

    if isinstance(template, list):
        if isinstance(candidate, list):
            cleaned = []
            for item in candidate:
                if isinstance(item, (str, int, float, bool)) or item is None:
                    cleaned.append(item)
                elif isinstance(item, dict):
                    cleaned.append(item)
                else:
                    cleaned.append(str(item))
            return cleaned
        return copy.deepcopy(template)

    if candidate is None:
        return copy.deepcopy(template)

    if isinstance(candidate, (str, int, float, bool)):
        return candidate

    return copy.deepcopy(template)


def is_empty(value: Any) -> bool:
    if value is None:
        return True

    if isinstance(value, str):
        return value.strip() == ""

    if isinstance(value, list):
        return len(value) == 0 or all(is_empty(v) for v in value)

    if isinstance(value, dict):
        return all(is_empty(v) for v in value.values())

    return False


def normalize_segment_id_list(
    values: Any,
    valid_segment_ids: List[str]
) -> List[str]:
    if not isinstance(values, list):
        return []

    order_map = {seg_id: i for i, seg_id in enumerate(valid_segment_ids)}
    valid_set = set(valid_segment_ids)

    cleaned = []
    seen = set()

    for value in values:
        seg_id = str(value)
        if seg_id in valid_set and seg_id not in seen:
            cleaned.append(seg_id)
            seen.add(seg_id)

    cleaned.sort(key=lambda x: order_map.get(x, 10**9))
    return cleaned


def postprocess_note(
    template: Dict[str, Any],
    llm_note: Dict[str, Any],
    valid_segment_ids: List[str],
) -> Dict[str, Any]:
    note = coerce_to_template(template, llm_note)

    # Clean evidence ids
    evidence = note.get("evidence", {})
    for key in [
        "subjective_segment_ids",
        "objective_segment_ids",
        "assessment_segment_ids",
        "plan_segment_ids",
    ]:
        evidence[key] = normalize_segment_id_list(evidence.get(key), valid_segment_ids)
    note["evidence"] = evidence

    # Make sure quality fields are lists
    quality = note.get("quality", {})
    if not isinstance(quality.get("missing_sections"), list):
        quality["missing_sections"] = []
    if not isinstance(quality.get("uncertain_fields"), list):
        quality["uncertain_fields"] = []

    # Auto-detect missing major sections
    auto_missing = []
    for section_name in ["subjective", "objective", "assessment", "plan"]:
        if section_name in note and is_empty(note[section_name]):
            auto_missing.append(section_name)

    quality["missing_sections"] = sorted(
        set([str(x) for x in quality["missing_sections"]] + auto_missing)
    )
    quality["uncertain_fields"] = [str(x) for x in quality["uncertain_fields"]]

    note["quality"] = quality
    return note


def nice_value(value: Any, fallback: str = "Not documented") -> str:
    if is_empty(value):
        return fallback
    if isinstance(value, list):
        return "; ".join(str(v) for v in value if not is_empty(v)) or fallback
    return str(value)


def render_soap_text(note: Dict[str, Any]) -> str:
    encounter = note.get("encounter", {})
    subjective = note.get("subjective", {})
    objective = note.get("objective", {})
    assessment = note.get("assessment", {})
    plan = note.get("plan", {})
    vitals = objective.get("vitals", {}) if isinstance(objective, dict) else {}

    lines = []

    lines.append("SOAP NOTE")
    lines.append("")
    lines.append("Encounter")
    lines.append(f"  encounter_id: {nice_value(encounter.get('encounter_id'))}")
    lines.append(f"  date: {nice_value(encounter.get('date'))}")
    lines.append(f"  clinician: {nice_value(encounter.get('clinician'))}")
    lines.append(f"  patient: {nice_value(encounter.get('patient'))}")
    lines.append("")

    lines.append("Subjective")
    lines.append(f"  chief complaint: {nice_value(subjective.get('chief_complaint'))}")
    lines.append(f"  HPI summary: {nice_value(subjective.get('hpi_summary'))}")
    lines.append(f"  medications mentioned: {nice_value(subjective.get('medications_mentioned'))}")
    lines.append(f"  allergies mentioned: {nice_value(subjective.get('allergies_mentioned'))}")
    lines.append(f"  relevant history: {nice_value(subjective.get('relevant_history'))}")
    lines.append(f"  patient reported symptoms: {nice_value(subjective.get('patient_reported_symptoms'))}")
    lines.append(f"  patient denied symptoms: {nice_value(subjective.get('patient_denied_symptoms'))}")
    lines.append("")

    lines.append("Objective")
    lines.append(
        "  vitals: "
        f"bp={nice_value(vitals.get('bp'))}, "
        f"hr={nice_value(vitals.get('hr'))}, "
        f"temp={nice_value(vitals.get('temp'))}, "
        f"rr={nice_value(vitals.get('rr'))}, "
        f"spo2={nice_value(vitals.get('spo2'))}, "
        f"weight={nice_value(vitals.get('weight'))}"
    )
    lines.append(f"  exam findings: {nice_value(objective.get('exam_findings'))}")
    lines.append(f"  tests/results discussed: {nice_value(objective.get('tests_or_results_discussed'))}")
    lines.append("")

    lines.append("Assessment")
    lines.append(f"  clinician impression: {nice_value(assessment.get('clinician_impression'))}")
    lines.append(f"  problems addressed: {nice_value(assessment.get('problems_addressed'))}")
    lines.append("")

    lines.append("Plan")
    lines.append(f"  medications started/changed: {nice_value(plan.get('medications_started_or_changed'))}")
    lines.append(f"  tests ordered: {nice_value(plan.get('tests_ordered'))}")
    lines.append(f"  referrals: {nice_value(plan.get('referrals'))}")
    lines.append(f"  patient instructions: {nice_value(plan.get('patient_instructions'))}")
    lines.append(f"  follow up: {nice_value(plan.get('follow_up'))}")
    lines.append(f"  return precautions: {nice_value(plan.get('return_precautions'))}")
    lines.append("")

    lines.append("Quality")
    quality = note.get("quality", {})
    lines.append(f"  missing sections: {nice_value(quality.get('missing_sections'))}")
    lines.append(f"  uncertain fields: {nice_value(quality.get('uncertain_fields'))}")
    lines.append(f"  notes: {nice_value(quality.get('notes'))}")

    return "\n".join(lines)


def build_evidence_preview(
    note: Dict[str, Any],
    normalized_segments: List[Dict[str, Any]]
) -> Dict[str, List[Dict[str, Any]]]:
    by_id = {seg["id"]: seg for seg in normalized_segments}

    section_map = {
        "Subjective": "subjective_segment_ids",
        "Objective": "objective_segment_ids",
        "Assessment": "assessment_segment_ids",
        "Plan": "plan_segment_ids",
    }

    preview = {}

    for display_name, evidence_key in section_map.items():
        rows = []
        for seg_id in note.get("evidence", {}).get(evidence_key, []):
            seg = by_id.get(str(seg_id))
            if not seg:
                continue
            rows.append(
                {
                    "id": seg["id"],
                    "speaker": seg["speaker"],
                    "start": seg["start"],
                    "end": seg["end"],
                    "text": seg["text"],
                }
            )
        preview[display_name] = rows

    return preview

In [5]:
# Cell 6: Quick health check for your local Ollama server

response = requests.get(OLLAMA_TAGS_URL, timeout=30)
response.raise_for_status()

ollama_models = response.json().get("models", [])
model_names = [m.get("name") for m in ollama_models]

print("Ollama is reachable.")
print("Installed local models:")
for name in model_names:
    print(" -", name)

if OLLAMA_MODEL not in model_names:
    print(f"\nWarning: {OLLAMA_MODEL!r} is not in your local model list.")
    print("Either change OLLAMA_MODEL above or pull the model locally before running generation.")
    print(f"Example terminal command: ollama pull {OLLAMA_MODEL}")

Ollama is reachable.
Installed local models:
 - mistral:7b
 - nomic-embed-text:latest
 - qwen2.5-coder:1.5b-base
 - llama3.1:8b
 - mistral:latest


# Cell 7: What input JSON shape this notebook expects

At minimum, the notebook expects a transcript JSON with a top-level list like:

{
  "segments": [
    {
      "id": "0",
      "start": 0.0,
      "end": 2.4,
      "speaker": "doctor",
      "text": "What brings you in today?"
    },
    {
      "id": "1",
      "start": 2.5,
      "end": 7.8,
      "speaker": "patient",
      "text": "I've had a sore throat for three days."
    }
  ]
}

The notebook also tries a few alternative speaker keys like:
- role
- label
- speaker_label
- classified_speaker

In [6]:
# Cell 8: Load the transcript JSON and the target template

transcript_payload = load_json(CLASSIFIED_TRANSCRIPT_PATH)
note_template = load_note_template(OPTIONAL_TEMPLATE_PATH)
note_template = fill_encounter_defaults(note_template, transcript_payload, CLASSIFIED_TRANSCRIPT_PATH)

normalized_segments = normalize_segments(transcript_payload)

print(f"Loaded transcript from: {CLASSIFIED_TRANSCRIPT_PATH}")
print(f"Normalized segment count: {len(normalized_segments)}")
print()

speaker_counts = Counter(seg["speaker"] for seg in normalized_segments)
print("Speaker counts:")
pprint(dict(speaker_counts))

print("\nFirst 5 normalized segments:")
pprint(normalized_segments[:5])

print("\nTemplate top-level keys:")
print(list(note_template.keys()))

Loaded transcript from: transcription_outputs/classified_segments.json
Normalized segment count: 33

Speaker counts:
{'doctor': 12, 'patient': 21}

First 5 normalized segments:
[{'end': 12.32,
  'id': '0',
  'speaker': 'doctor',
  'start': 0.0,
  'text': "So, hello. My name is Daniela Gonzalez. I'm a second year medical "
          "student and this is my colleague, Napalia Flores, she's also a "
          'medical student.'},
 {'end': 16.96,
  'id': '1',
  'speaker': 'doctor',
  'start': 12.32,
  'text': "And today we're just gonna find out what brought you in here "
          'today.'},
 {'end': 20.64,
  'id': '2',
  'speaker': 'doctor',
  'start': 16.96,
  'text': 'So, is your name Edith Riswell?'},
 {'end': 21.44,
  'id': '3',
  'speaker': 'patient',
  'start': 20.64,
  'text': 'Yes, it is.'},
 {'end': 22.96,
  'id': '4',
  'speaker': 'doctor',
  'start': 21.44,
  'text': 'Yes, how would you like me to refer you?'}]

Template top-level keys:
['schema_version', 'note_type', 'encount

In [7]:
# Cell 9: Turn the normalized segments into the text we will send to the model

transcript_text = build_transcript_for_llm(
    normalized_segments,
    max_chars=MAX_TRANSCRIPT_CHARS,
)

print(f"Transcript text length sent to model: {len(transcript_text):,} characters")
print("\nTranscript preview:\n")
print(transcript_text[:2500])

Transcript text length sent to model: 4,483 characters

Transcript preview:

[segment_id=0 | speaker=doctor | start=0.00 | end=12.32] So, hello. My name is Daniela Gonzalez. I'm a second year medical student and this is my colleague, Napalia Flores, she's also a medical student.
[segment_id=1 | speaker=doctor | start=12.32 | end=16.96] And today we're just gonna find out what brought you in here today.
[segment_id=2 | speaker=doctor | start=16.96 | end=20.64] So, is your name Edith Riswell?
[segment_id=3 | speaker=patient | start=20.64 | end=21.44] Yes, it is.
[segment_id=4 | speaker=doctor | start=21.44 | end=22.96] Yes, how would you like me to refer you?
[segment_id=5 | speaker=patient | start=22.96 | end=23.84] Edith is fine.
[segment_id=6 | speaker=doctor | start=23.84 | end=27.60] Edith is fine? Alright. So, Edith, what brings you in here today?
[segment_id=7 | speaker=patient | start=28.40 | end=38.00] I'm just, I'm just feeling terrible and I'm losing too much weight for no rea

In [8]:
# Cell 10: Build the structured-output schema and the charting prompt

json_schema = template_to_json_schema(note_template)
charting_prompt = build_charting_prompt(note_template, transcript_text)

print("JSON schema preview:")
pprint(json_schema)

print("\nPrompt preview:\n")
print(charting_prompt[:3000])

JSON schema preview:
{'additionalProperties': False,
 'properties': {'assessment': {'additionalProperties': False,
                               'properties': {'clinician_impression': {'items': {'type': ['string',
                                                                                          'number',
                                                                                          'integer',
                                                                                          'boolean',
                                                                                          'null',
                                                                                          'object']},
                                                                       'type': 'array'},
                                              'problems_addressed': {'items': {'type': ['string',
                                                                                        'number'

In [10]:
# Cell 11: Call the local Ollama model to generate the chart note draft

ollama_result = call_ollama_generate_json(
    model=OLLAMA_MODEL,
    prompt=charting_prompt,
    json_schema=json_schema,
    timeout_seconds=1200,
)

raw_ollama_response = ollama_result["raw_ollama_response"]
llm_note_draft = ollama_result["parsed_json"]

print("Raw Ollama response keys:")
print(list(raw_ollama_response.keys()))

print("\nModel draft JSON:")
pprint(llm_note_draft)

Raw Ollama response keys:
['model', 'created_at', 'response', 'done', 'done_reason', 'context', 'total_duration', 'load_duration', 'prompt_eval_count', 'prompt_eval_duration', 'eval_count', 'eval_duration']

Model draft JSON:
{'assessment': {'clinician_impression': ['Possible unexplained weight loss and '
                                         'fatigue'],
                'problems_addressed': ['Tiredness', 'Weight loss']},
 'encounter': {'clinician': 'Daniela Gonzalez',
               'date': None,
               'encounter_id': 'classified_segments',
               'patient': 'Edith Riswell'},
 'evidence': {'assessment_segment_ids': [],
              'objective_segment_ids': [],
              'plan_segment_ids': [],
              'subjective_segment_ids': [0,
                                         1,
                                         2,
                                         4,
                                         5,
                                         6,
       

In [11]:
# Cell 12: Postprocess the model draft so it exactly matches our template shape

valid_segment_ids = [seg["id"] for seg in normalized_segments]

final_note = postprocess_note(
    template=note_template,
    llm_note=llm_note_draft,
    valid_segment_ids=valid_segment_ids,
)

print("Final structured note:")
pprint(final_note)

Final structured note:
{'assessment': {'clinician_impression': ['Possible unexplained weight loss and '
                                         'fatigue'],
                'problems_addressed': ['Tiredness', 'Weight loss']},
 'encounter': {'clinician': 'Daniela Gonzalez',
               'date': None,
               'encounter_id': 'classified_segments',
               'patient': 'Edith Riswell'},
 'evidence': {'assessment_segment_ids': [],
              'objective_segment_ids': [],
              'plan_segment_ids': [],
              'subjective_segment_ids': ['0',
                                         '1',
                                         '2',
                                         '4',
                                         '5',
                                         '6',
                                         '7',
                                         '8',
                                         '9',
                                         '10',
             

In [12]:
# Cell 13: Render a readable text version of the note

note_text = render_soap_text(final_note)

print(note_text)

SOAP NOTE

Encounter
  encounter_id: classified_segments
  date: Not documented
  clinician: Daniela Gonzalez
  patient: Edith Riswell

Subjective
  chief complaint: Feeling terrible and losing weight for no reason
  HPI summary: Patient reports feeling tired, ill, and losing significant amount of weight (30-35 lbs) over the past six to eight months. No pain, bruising, or skipping meals reported.
  medications mentioned: Not documented
  allergies mentioned: Not documented
  relevant history: Not documented
  patient reported symptoms: Tiredness; Feeling ill; Weight loss
  patient denied symptoms: Not documented

Objective
  vitals: bp=Not documented, hr=Not documented, temp=Not documented, rr=Not documented, spo2=Not documented, weight=Not documented
  exam findings: Not documented
  tests/results discussed: Not documented

Assessment
  clinician impression: Possible unexplained weight loss and fatigue
  problems addressed: Tiredness; Weight loss

Plan
  medications started/changed: N

In [13]:
# Cell 14: Build a simple evidence preview by section

evidence_preview = build_evidence_preview(final_note, normalized_segments)

for section_name, rows in evidence_preview.items():
    print("=" * 80)
    print(section_name)
    print("=" * 80)

    if not rows:
        print("No evidence rows recorded.\n")
        continue

    for row in rows:
        print(
            f"[id={row['id']} | speaker={row['speaker']} | "
            f"{row['start']:.2f}-{row['end']:.2f}] {row['text']}"
        )
    print()

Subjective
[id=0 | speaker=doctor | 0.00-12.32] So, hello. My name is Daniela Gonzalez. I'm a second year medical student and this is my colleague, Napalia Flores, she's also a medical student.
[id=1 | speaker=doctor | 12.32-16.96] And today we're just gonna find out what brought you in here today.
[id=2 | speaker=doctor | 16.96-20.64] So, is your name Edith Riswell?
[id=4 | speaker=doctor | 21.44-22.96] Yes, how would you like me to refer you?
[id=5 | speaker=patient | 22.96-23.84] Edith is fine.
[id=6 | speaker=doctor | 23.84-27.60] Edith is fine? Alright. So, Edith, what brings you in here today?
[id=7 | speaker=patient | 28.40-38.00] I'm just, I'm just feeling terrible and I'm losing too much weight for no reason.
[id=8 | speaker=doctor | 38.64-41.04] Oh, right. Can you tell me more?
[id=9 | speaker=patient | 42.40-52.08] Yeah, it's been probably, oh I don't know, maybe six or eight months that I've just been feeling
[id=10 | speaker=patient | 53.04-60.64] more and more tired. It's

In [15]:
# Cell 15: Save outputs to disk

output_stem = CLASSIFIED_TRANSCRIPT_PATH.stem + "_soap_note"

json_output_path = OUTPUT_DIR / f"{output_stem}.json"
text_output_path = OUTPUT_DIR / f"{output_stem}.txt"
raw_output_path = OUTPUT_DIR / f"{output_stem}_raw_ollama_response.json"

save_json(json_output_path, final_note)
save_json(raw_output_path, raw_ollama_response)

with open(text_output_path, "w", encoding="utf-8") as f:
    f.write(note_text)

print("Saved files:")
print(" -", json_output_path.resolve())
print(" -", text_output_path.resolve())
print(" -", raw_output_path.resolve())

Saved files:
 - /home/deuterium/whisper_app/soap_note_outputs/classified_segments_soap_note.json
 - /home/deuterium/whisper_app/soap_note_outputs/classified_segments_soap_note.txt
 - /home/deuterium/whisper_app/soap_note_outputs/classified_segments_soap_note_raw_ollama_response.json


# Cell 16: How to interpret the outputs

1. `final_note`
   This is the main structured SOAP-style JSON object.
   Think of this as the machine-readable chart note.

2. `note_text`
   This is just a human-friendly rendering of the same structured note.
   It is useful for quick review.

3. `evidence`
   These are section-level segment ids that support each major SOAP section.
   This is not perfect provenance at the field level, but it is a simple and useful first step.

4. `quality.missing_sections`
   These are major sections that were mostly unsupported by the transcript.

5. `quality.uncertain_fields`
   These are fields the model believed were mentioned but not clearly enough to chart confidently.

6. `raw_ollama_response`
   This is useful when debugging the model call itself.
   It can help you inspect model metadata and timing if something looks off.

Good next extensions:
- add field-level evidence, not just section-level evidence
- chunk long transcripts and merge notes
- add a second local validation pass with Ollama
- compare doctor-only vs patient-only context windows
- swap in a different chart template from disk